# Trabalho Prático 04 - Jogo dos Oito (Busca Cega e Informada)

**Disciplina:** GCC 128 - Inteligência Artificial  
**Professor:** Ahmed Ali Abdalla Esmin  
**Alunos:** Daniel de Jesus Moreira, João Guilherme Santos Ribeiro

---

## Introdução

Este notebook apresenta a resolução do **Jogo dos Oito**, um problema clássico de busca em espaço de estados.

O objetivo é transformar um estado inicial em um estado final desejado movimentando o espaço vazio do tabuleiro. Para isso, comparamos duas estratégias:

- **Busca em Largura (BFS):** busca cega que explora os estados por nível;
- **Busca A\*:** busca informada que utiliza uma heurística para priorizar estados mais promissores.

Ao final, avaliamos a quantidade de movimentos, o número de nós explorados e o tempo de execução de cada abordagem.

## 1. Definição do Problema e Funções Auxiliares

Nesta etapa definimos o estado objetivo do jogo, a forma de impressão do tabuleiro e a função responsável por gerar os estados vizinhos.

O tabuleiro é representado por uma tupla com 9 posições, e o valor `0` representa o espaço vazio que pode ser movimentado para cima, baixo, esquerda ou direita.

In [7]:
import time
from collections import deque

OBJETIVO = (1, 2, 3, 8, 0, 4, 7, 6, 5)

def imprimir_tabuleiro(estado):
    """Formata a tupla para impressão em formato 3x3 para valorizar a interface."""
    for i in range(0, 9, 3):
        linha = [str(x) if x != 0 else ' ' for x in estado[i:i+3]]
        print(f"[{linha[0]}] [{linha[1]}] [{linha[2]}]")
    print("-" * 13)

def obter_vizinhos(estado):
    """Gera os próximos estados possíveis movendo o espaço vazio (0)."""
    vizinhos = []
    idx_vazio = estado.index(0)
    linha, coluna = divmod(idx_vazio, 3)

    movimentos = {
        'Cima': -3,
        'Baixo': 3,
        'Esquerda': -1,
        'Direita': 1
    }

    if linha > 0: vizinhos.append(('Cima', idx_vazio + movimentos['Cima']))
    if linha < 2: vizinhos.append(('Baixo', idx_vazio + movimentos['Baixo']))
    if coluna > 0: vizinhos.append(('Esquerda', idx_vazio + movimentos['Esquerda']))
    if coluna < 2: vizinhos.append(('Direita', idx_vazio + movimentos['Direita']))

    estados_gerados = []
    for acao, novo_idx in vizinhos:
        novo_estado = list(estado)
        novo_estado[idx_vazio], novo_estado[novo_idx] = novo_estado[novo_idx], novo_estado[idx_vazio]
        estados_gerados.append((acao, tuple(novo_estado)))

    return estados_gerados

## 2. Busca Cega em Largura (BFS)

A **Busca em Largura** explora primeiro todos os estados que estão a uma mesma profundidade antes de avançar para o próximo nível.

Como cada movimento possui o mesmo custo, a BFS encontra uma solução com número mínimo de passos. Porém, ela pode consumir mais memória, pois precisa manter muitos estados na fila de exploração.

In [8]:
def busca_em_largura(estado_inicial):
    """
    Implementa a Busca em Largura (BFS).
    Retorna o caminho de ações, a quantidade de nós explorados e o tempo gasto.
    """
    inicio_tempo = time.time()
    
    fila = deque([(estado_inicial, [])])
    
    visitados = set()
    visitados.add(estado_inicial)
    
    nos_explorados = 0

    while fila:
        estado_atual, caminho = fila.popleft()
        nos_explorados += 1

        if estado_atual == OBJETIVO:
            tempo_total = time.time() - inicio_tempo
            return caminho, nos_explorados, tempo_total

        for acao, novo_estado in obter_vizinhos(estado_atual):
            if novo_estado not in visitados:
                visitados.add(novo_estado)
                fila.append((novo_estado, caminho + [acao]))

    return None, nos_explorados, time.time() - inicio_tempo

## 3. Execução da Busca em Largura

A seguir definimos um estado inicial de exemplo e executamos a BFS.

Além do caminho encontrado, são exibidos o tempo de execução, a quantidade de nós explorados e o passo a passo da solução no tabuleiro.

In [9]:
ESTADO_INICIAL_EXEMPLO = (2, 0, 3, 1, 7, 4, 6, 8, 5)

print("--- ESTADO INICIAL ---")
imprimir_tabuleiro(ESTADO_INICIAL_EXEMPLO)

print("Iniciando Busca Cega (Largura)... Aguarde.")
caminho_bfs, nos_bfs, tempo_bfs = busca_em_largura(ESTADO_INICIAL_EXEMPLO)

if caminho_bfs is not None:
    print(f"Solução encontrada em {tempo_bfs:.4f} segundos!")
    print(f"Nós explorados na memória: {nos_bfs}")
    print(f"Quantidade de movimentos: {len(caminho_bfs)}")
    print(f"Passo a passo: {caminho_bfs}\n")
    
    estado_temp = ESTADO_INICIAL_EXEMPLO
    for passo, acao in enumerate(caminho_bfs, 1):
        print(f"Passo {passo}: Mover espaço para '{acao}'")
        for a, e in obter_vizinhos(estado_temp):
            if a == acao:
                estado_temp = e
                break
        imprimir_tabuleiro(estado_temp)
else:
    print("Nenhuma solução foi encontrada.")

--- ESTADO INICIAL ---
[2] [ ] [3]
[1] [7] [4]
[6] [8] [5]
-------------
Iniciando Busca Cega (Largura)... Aguarde.
Solução encontrada em 0.0030 segundos!
Nós explorados na memória: 159
Quantidade de movimentos: 7
Passo a passo: ['Esquerda', 'Baixo', 'Direita', 'Baixo', 'Esquerda', 'Cima', 'Direita']

Passo 1: Mover espaço para 'Esquerda'
[ ] [2] [3]
[1] [7] [4]
[6] [8] [5]
-------------
Passo 2: Mover espaço para 'Baixo'
[1] [2] [3]
[ ] [7] [4]
[6] [8] [5]
-------------
Passo 3: Mover espaço para 'Direita'
[1] [2] [3]
[7] [ ] [4]
[6] [8] [5]
-------------
Passo 4: Mover espaço para 'Baixo'
[1] [2] [3]
[7] [8] [4]
[6] [ ] [5]
-------------
Passo 5: Mover espaço para 'Esquerda'
[1] [2] [3]
[7] [8] [4]
[ ] [6] [5]
-------------
Passo 6: Mover espaço para 'Cima'
[1] [2] [3]
[ ] [8] [4]
[7] [6] [5]
-------------
Passo 7: Mover espaço para 'Direita'
[1] [2] [3]
[8] [ ] [4]
[7] [6] [5]
-------------


## 4. Heurística da Distância de Manhattan

Para aplicar uma busca informada, utilizamos a **Distância de Manhattan** como heurística.

Essa heurística soma, para cada peça, a distância horizontal e vertical entre sua posição atual e sua posição no estado objetivo. Quanto menor esse valor, mais próximo o tabuleiro está da solução.

In [10]:
def heuristica_manhattan(estado):
    """
    Calcula a Distância de Manhattan entre o estado atual e o OBJETIVO.
    Conta quantos 'passos' (horizontal + vertical) cada peça está do seu lugar correto.
    """
    distancia_total = 0
    for i, valor in enumerate(estado):
        if valor != 0:
            idx_objetivo = OBJETIVO.index(valor)
            
            linha_atual, col_atual = divmod(i, 3)
            
            linha_obj, col_obj = divmod(idx_objetivo, 3)
            
            distancia_total += abs(linha_atual - linha_obj) + abs(col_atual - col_obj)
            
    return distancia_total

## 5. Busca Informada A\*

O algoritmo **A\*** combina o custo já percorrido com uma estimativa do custo restante até o objetivo.

A prioridade de cada estado é calculada por:

$$f(n) = g(n) + h(n)$$

em que `g(n)` representa o número de movimentos já realizados e `h(n)` representa a heurística de Manhattan. Assim, o algoritmo tende a explorar primeiro os estados mais promissores.

In [11]:
import heapq

def busca_a_estrela(estado_inicial):
    """
    Implementa a Busca Informada A* (A-Star).
    Retorna o caminho de ações, a quantidade de nós explorados e o tempo gasto.
    """
    inicio_tempo = time.time()
    
    g_score_inicial = 0
    
    h_score_inicial = heuristica_manhattan(estado_inicial)
    
    f_score_inicial = g_score_inicial + h_score_inicial
    
    contador = 0
    
    fila = [(f_score_inicial, contador, g_score_inicial, estado_inicial, [])]
    
    melhor_g_score = {estado_inicial: 0}
    
    nos_explorados = 0

    while fila:
        _, _, g_atual, estado_atual, caminho = heapq.heappop(fila)
        nos_explorados += 1

        if estado_atual == OBJETIVO:
            tempo_total = time.time() - inicio_tempo
            return caminho, nos_explorados, tempo_total

        for acao, novo_estado in obter_vizinhos(estado_atual):
            novo_g = g_atual + 1 
            
            if novo_estado not in melhor_g_score or novo_g < melhor_g_score[novo_estado]:
                melhor_g_score[novo_estado] = novo_g
                
                novo_h = heuristica_manhattan(novo_estado)
                novo_f = novo_g + novo_h
                
                contador += 1
                heapq.heappush(fila, (novo_f, contador, novo_g, novo_estado, caminho + [acao]))

    return None, nos_explorados, time.time() - inicio_tempo

## 6. Execução do A\* e Comparação dos Resultados

Nesta etapa executamos o A\* sobre o mesmo estado inicial usado na BFS.

A comparação considera a quantidade de movimentos necessários para chegar ao objetivo e o número de nós explorados por cada algoritmo.

In [12]:
print("--- TESTE BUSCA INFORMADA (A*) ---")
print("Iniciando A*... Aguarde.")
caminho_astar, nos_astar, tempo_astar = busca_a_estrela(ESTADO_INICIAL_EXEMPLO)

if caminho_astar is not None:
    print(f"Solução encontrada em {tempo_astar:.4f} segundos!")
    print(f"Nós explorados na memória: {nos_astar}")
    print(f"Quantidade de movimentos: {len(caminho_astar)}")
    print(f"Passo a passo: {caminho_astar}\n")
    
    estado_temp = ESTADO_INICIAL_EXEMPLO
    for passo, acao in enumerate(caminho_astar, 1):
        print(f"Passo {passo}: Mover espaço para '{acao}'")
        for a, e in obter_vizinhos(estado_temp):
            if a == acao:
                estado_temp = e
                break
        imprimir_tabuleiro(estado_temp)
        
    print("=========================================")
    print("      COMPARAÇÃO DE DESEMPENHO           ")
    print("=========================================")
    print(f"Movimentos até a vitória: BFS = {len(caminho_bfs)} | A* = {len(caminho_astar)}")
    
    print(f"Nós explorados (BFS): {nos_bfs}")
    print(f"Nós explorados (A*):  {nos_astar}")
    
    if nos_bfs > 0:
        economia = ((nos_bfs - nos_astar) / nos_bfs) * 100
        print(f"\nConclusão: O A* reduziu o número de nós explorados em {economia:.1f}%!")
else:
    print("Nenhuma solução foi encontrada.")

--- TESTE BUSCA INFORMADA (A*) ---
Iniciando A*... Aguarde.
Solução encontrada em 0.0000 segundos!
Nós explorados na memória: 8
Quantidade de movimentos: 7
Passo a passo: ['Esquerda', 'Baixo', 'Direita', 'Baixo', 'Esquerda', 'Cima', 'Direita']

Passo 1: Mover espaço para 'Esquerda'
[ ] [2] [3]
[1] [7] [4]
[6] [8] [5]
-------------
Passo 2: Mover espaço para 'Baixo'
[1] [2] [3]
[ ] [7] [4]
[6] [8] [5]
-------------
Passo 3: Mover espaço para 'Direita'
[1] [2] [3]
[7] [ ] [4]
[6] [8] [5]
-------------
Passo 4: Mover espaço para 'Baixo'
[1] [2] [3]
[7] [8] [4]
[6] [ ] [5]
-------------
Passo 5: Mover espaço para 'Esquerda'
[1] [2] [3]
[7] [8] [4]
[ ] [6] [5]
-------------
Passo 6: Mover espaço para 'Cima'
[1] [2] [3]
[ ] [8] [4]
[7] [6] [5]
-------------
Passo 7: Mover espaço para 'Direita'
[1] [2] [3]
[8] [ ] [4]
[7] [6] [5]
-------------
      COMPARAÇÃO DE DESEMPENHO           
Movimentos até a vitória: BFS = 7 | A* = 7
Nós explorados (BFS): 159
Nós explorados (A*):  8

Conclusão: O A*

## 7. Conclusão

A BFS e o A\* encontraram uma solução com a mesma quantidade de movimentos, demonstrando que ambos chegaram a um caminho ótimo para este caso.

A diferença principal está na eficiência: com o uso da heurística de Manhattan, o A\* explorou muito menos nós que a BFS. Isso mostra a vantagem de utilizar informação heurística em problemas de busca, principalmente quando o espaço de estados cresce.